# Arena 3DGS — 91-Image Reconstruction

Uses a pre-computed COLMAP model (global mapper, 91/91 images registered, 18.5K points).
Upload the zip file created by `scripts/export_3dgs_input.py`.

**Steps:**
| Step | Cell | What | Time | GPU? |
|------|------|------|------|------|
| 1 | **Cell 1** | Mount Drive + start/continue session | 30s | No |
| 2 | **Cell 2** | Install deps (COLMAP, PyTorch, gsplat) | 3 min | No* |
| 3 | **Cell 3** | Upload & extract COLMAP zip | 1 min | No |
| 4 | **Cell 4** | Convert to 3DGS format | 1 min | No |
| 5 | **Cell 5** | Train 3DGS (30K iters, ~30 min) | 30 min | **Yes (T4+)** |
| 6 | **Cell 6** | Export PLY + validate + compress | 1 min | No |

---
## Setup
---

In [ ]:
#@title === 1. Mount Drive + Session Management ===
import os, json, datetime

DRIVE_PATH = "/content/drive/MyDrive/arena_3dgs"
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_PATH}")

# ── Session state ──
SESSION_PATH = os.path.join(DRIVE_PATH, "session_state.json")

def load_session():
    if os.path.exists(SESSION_PATH):
        with open(SESSION_PATH) as f:
            return json.load(f)
    return {"steps": {}, "params": {}, "created_at": None, "updated_at": None}

def save_session(session):
    session["updated_at"] = datetime.datetime.now().isoformat()
    with open(SESSION_PATH, "w") as f:
        json.dump(session, f, indent=2)
    return session

def is_step_done(name):
    return session["steps"].get(name, False)

def mark_step(name):
    session["steps"][name] = True
    save_session(session)
    print(f"  [session] Step '{name}' completed")

def set_param(key, value):
    session["params"][key] = value
    save_session(session)

def get_param(key, default=None):
    return session["params"].get(key, default)

def checkpoint_path(name):
    return os.path.join(DRIVE_PATH, f"_{name}.marker")

# ── Fresh or continue? ──
session = load_session()
if session["created_at"] is None:
    session["created_at"] = datetime.datetime.now().isoformat()
    save_session(session)
    print("\n🆕 Starting fresh session.")
else:
    done_steps = [k for k, v in session["steps"].items() if v]
    print(f"\n📋 Existing session found with {len(done_steps)} completed step(s):")
    for s in done_steps:
        print(f"    ✓ {s}")
    print("\nContinue from where you left off, or reset with the button below.")

RESET = False  #@param {type:"boolean"}
if RESET:
    session = {"steps": {}, "params": {}, "created_at": datetime.datetime.now().isoformat(), "updated_at": None}
    save_session(session)
    print("\n🔄 Session reset. All steps will re-run.")

In [ ]:
#@title === 2. Install Dependencies (~3 min, idempotent) ===
import os, subprocess, atexit, sys

colmap_available = bool(os.popen("which colmap 2>/dev/null").read().strip())
skip_deps = is_step_done("deps_installed") and colmap_available

if skip_deps:
    print("Dependencies already installed (session state). Skipping.")
else:
    if is_step_done("deps_installed"):
        print("Session says deps installed but colmap not found. Re-installing.")

    DRV = checkpoint_path("deps_installed")
    colmap_missing = not colmap_available
    if not os.path.exists(DRV) or colmap_missing:
        print("[1/4] Installing COLMAP + display deps...")
        !apt-get update -qq && apt-get install -y -qq colmap xvfb libgl1-mesa-glx libglib2.0-0
        !colmap version 2>&1 | head -1

        print("[2/4] Installing PyTorch...")
        !pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118 -q
        import torch
        cuda_info = f"CUDA: {torch.cuda.is_available()}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB" if torch.cuda.is_available() else "CUDA: False"
        print(f"  PyTorch {torch.__version__}, {cuda_info}")

        print("[3/4] Installing Python packages...")
        !pip install plyfile numpy pillow opencv-python-headless tqdm gsplat scipy -q

        print("[4/4] Verifying GPU...")
        !touch "{DRV}"
        print("\nSystem deps installed!")
    else:
        print("System deps already installed (Drive marker found).")

    # Download training script
    import urllib.request
    SCRIPTS_DIR = "/content/scripts"
    os.makedirs(SCRIPTS_DIR, exist_ok=True)
    ENHANCED_PY = os.path.join(SCRIPTS_DIR, "train_3dgs_enhanced.py")
    if not os.path.exists(ENHANCED_PY):
        print("Downloading enhanced training script from GitHub...")
        url = ("https://raw.githubusercontent.com/"
               "kaarthik-balakrishnan/arena-3dgs/main/scripts/train_3dgs_enhanced.py")
        urllib.request.urlretrieve(url, ENHANCED_PY)
        print(f"  Downloaded {ENHANCED_PY} ({os.path.getsize(ENHANCED_PY)/1024:.0f} KB)")
    else:
        print(f"Training script already exists.")
    sys.path.insert(0, SCRIPTS_DIR)

    save_session(session)
    mark_step("deps_installed")

---
## Step 3: Upload COLMAP Data (~1 min)

Upload the zip file (`arena_3dgs_input.zip`) produced by `scripts/export_3dgs_input.py`.
It contains:
- 91 images (PINHOLE camera model, already converted from SIMPLE_RADIAL)
- `sparse/0/cameras.txt`, `images.txt`, `points3D.txt` (.bin and .ply also)
- 18,554 sparse 3D points

**How to upload:** Colab left sidebar → 📁 Files → upload `arena_3dgs_input.zip` to `/content/`

If already uploaded, this cell detects and extracts it.
---

In [ ]:
#@title === 3. Upload & Extract COLMAP Data (~1 min, idempotent) ===
import os, zipfile, glob, shutil

if is_step_done("colmap_extracted"):
    print("COLMAP data already extracted (session state). Skipping.")
else:
    INPUT_DIR = "/content/gaussian-splatting/input"
    SPARSE_DIR = os.path.join(INPUT_DIR, "sparse", "0")
    os.makedirs(SPARSE_DIR, exist_ok=True)

    # Check if already extracted from a previous run
    img_txt = os.path.join(SPARSE_DIR, "images.txt")
    if os.path.exists(img_txt):
        with open(img_txt) as f:
            n = sum(1 for l in f if l.strip() and not l.startswith('#')) // 2
        print(f"Found existing model: {n} images")
        if n >= 80:
            mark_step("colmap_extracted")
        else:
            print(f"Only {n} images — looking for 91-image zip...")

    if not is_step_done("colmap_extracted"):
        # Search for zip in Colab's upload locations
        candidates = glob.glob("/content/*.zip") + glob.glob("/tmp/*.zip") + glob.glob("*.zip")
        if not candidates:
            raise FileNotFoundError(
                "No .zip file found in /content/.\n"
                "Upload arena_3dgs_input.zip via Colab left sidebar → 📁 Files → upload to /content/"
            )

        zip_path = candidates[0]
        print(f"Extracting {os.path.basename(zip_path)} ({os.path.getsize(zip_path)/1024/1024:.0f} MB)...")

        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(INPUT_DIR)

        # Handle nested directory: zip has arena_3dgs_input/ prefix
        nested = os.path.join(INPUT_DIR, "arena_3dgs_input")
        if os.path.exists(nested):
            for item in os.listdir(nested):
                shutil.move(os.path.join(nested, item), os.path.join(INPUT_DIR, item))
            os.rmdir(nested)
            print("  Flattened nested directory.")

        # Make sure images are where 3DGS expects them
        images_dir = os.path.join(INPUT_DIR, "images")
        if not os.path.exists(images_dir) or len(os.listdir(images_dir)) < 80:
            # Search for images in subdirs
            for root, dirs, files in os.walk(INPUT_DIR):
                jpgs = [f for f in files if f.lower().endswith('.jpg')]
                if len(jpgs) >= 80:
                    os.makedirs(images_dir, exist_ok=True)
                    for f in jpgs:
                        shutil.move(os.path.join(root, f), os.path.join(images_dir, f))
                    print(f"  Moved {len(jpgs)} images to input/images/")
                    break

        # Verify
        with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
            n = sum(1 for l in f if l.strip() and not l.startswith('#')) // 2
        pts_file = os.path.join(SPARSE_DIR, "points3D.txt")
        n_pts = 0
        if os.path.exists(pts_file):
            n_pts = sum(1 for l in open(pts_file) if l.strip() and not l.startswith('#'))
        img_count = len(os.listdir(os.path.join(INPUT_DIR, "images"))) if os.path.exists(os.path.join(INPUT_DIR, "images")) else 0

        print(f"\n  ✓ {n} registered images")
        print(f"  ✓ {n_pts} sparse points")
        print(f"  ✓ {img_count} image files")

        if n < 80:
            raise RuntimeError(f"Expected 90+ images in model, got {n}. Wrong zip?")

        set_param("training_images", n)
        mark_step("colmap_extracted")

---
## Step 4: Convert to 3DGS Format (~1 min)

Our model is already in PINHOLE format and has binary files. This cell
verifies the setup and rebuilds binary files if needed.
---

In [ ]:
#@title === 4. Convert to 3DGS Format (~1 min, idempotent) ===
import os, glob, subprocess, shutil

if is_step_done("data_converted"):
    print("Data already converted (session state). Skipping.")
else:
    if not os.popen("which colmap 2>/dev/null").read().strip():
        raise RuntimeError("colmap not found. Run Cell 2 first.")

    INPUT_DIR = "/content/gaussian-splatting/input"
    SPARSE_DIR = os.path.join(INPUT_DIR, "sparse", "0")
    images_dir = os.path.join(INPUT_DIR, "images")

    # Verify required files
    required = ["cameras.txt", "images.txt", "points3D.txt", "cameras.bin", "images.bin", "points3D.bin"]
    missing = [f for f in required if not os.path.exists(os.path.join(SPARSE_DIR, f))]
    if missing:
        print(f"Missing files: {missing}")
        print("Rebuilding from text...")
        bin_dir = "/content/gaussian-splatting/sparse_bin"
        os.makedirs(bin_dir, exist_ok=True)
        result = subprocess.run(
            ["colmap", "model_converter",
             "--input_path", SPARSE_DIR,
             "--output_path", bin_dir,
             "--output_type", "BIN"],
            capture_output=True, text=True
        )
        if os.path.exists(os.path.join(bin_dir, "images.bin")):
            for fn in ["cameras.bin", "images.bin", "points3D.bin"]:
                shutil.copy2(os.path.join(bin_dir, fn), os.path.join(SPARSE_DIR, fn))
            shutil.rmtree(bin_dir, ignore_errors=True)
            print("  Built binary files.")
        else:
            print(f"WARNING: Binary conversion failed: {result.stderr}")
            print("This is OK — 3DGS can also read text format.")
    else:
        print("All required files present.")

    # Verify camera model is PINHOLE
    with open(os.path.join(SPARSE_DIR, "cameras.txt")) as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                parts = line.split()
                is_pinhole = parts[1] == "PINHOLE"
                break
    print(f"Camera model: {'PINHOLE ✓' if is_pinhole else 'SIMPLE_RADIAL (will convert)'}")

    if not is_pinhole:
        cam_path = os.path.join(SPARSE_DIR, "cameras.txt")
        with open(cam_path) as f:
            lines = f.readlines()
        with open(cam_path, 'w') as f:
            for line in lines:
                if line.startswith('#') or not line.strip():
                    f.write(line)
                else:
                    parts = line.strip().split()
                    if parts[1] == "SIMPLE_RADIAL":
                        f.write(f"{parts[0]} PINHOLE {parts[2]} {parts[3]} {parts[4]} {parts[4]} {parts[5]} {parts[6]}\n")
                    else:
                        f.write(line)
        print("  Converted SIMPLE_RADIAL → PINHOLE")

    # Organize images if needed
    if os.path.exists(images_dir) and len(os.listdir(images_dir)) >= 80:
        print(f"Images already organized: {len(os.listdir(images_dir))} files")
    else:
        os.makedirs(images_dir, exist_ok=True)
        count = 0
        for ext in ['*.jpg', '*.jpeg', '*.png']:
            for f in glob.glob(os.path.join(INPUT_DIR, ext)):
                os.rename(f, os.path.join(images_dir, os.path.basename(f)))
                count += 1
        print(f"Moved {count} images to input/images/")

    # Final verification
    with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
        num_images = sum(1 for l in f if l.strip() and not l.startswith('#')) // 2
    print(f"\nReady for training: {num_images} images, PINHOLE model")
    set_param("training_images", num_images)
    mark_step("data_converted")

---
## Step 5: Train 3D Gaussian Splatting

Requires GPU (T4 or better). Training uses gsplat for 2x faster rendering.
---

In [ ]:
#@title === 5A: Quick Test (~7 min) ===
import os, sys

if is_step_done("training_quick"):
    print("Quick test already done (session state). Skipping.")
else:
    if 'scripts.train_3dgs_enhanced' in sys.modules:
        del sys.modules['scripts.train_3dgs_enhanced']
    from scripts.train_3dgs_enhanced import train as train_3dgs
    from argparse import Namespace

    INPUT_DIR = "/content/gaussian-splatting/input"
    OUTPUT_DIR = "/content/gaussian-splatting/output/arena_3dgs"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    args = Namespace(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        iterations=3000,
        max_gaussians=100000,
        log_interval=500,
        max_res=800,
    )
    try:
        train_3dgs(args)
        print("\nQuick test complete!")
        mark_step("training_quick")
    except Exception as e:
        print(f"\nERROR: Training failed: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
#@title === 5B: Full Training 30K (~30 min, single run) ===
import os, sys

if is_step_done("training_30k"):
    print("Full training already done (session state). Skipping.")
else:
    if 'scripts.train_3dgs_enhanced' in sys.modules:
        del sys.modules['scripts.train_3dgs_enhanced']
    from scripts.train_3dgs_enhanced import train as train_3dgs
    from argparse import Namespace

    INPUT_DIR = "/content/gaussian-splatting/input"
    OUTPUT_DIR = "/content/gaussian-splatting/output/arena_3dgs"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    args = Namespace(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        iterations=30000,
        max_gaussians=500000,
        log_interval=1000,
        max_res=1600,
    )
    try:
        train_3dgs(args)
        print("\nFull training complete!")
        mark_step("training_30k")
    except Exception as e:
        print(f"\nERROR: Training failed: {e}")
        import traceback
        traceback.print_exc()

---
## Step 6: Export & Compress
---

In [ ]:
#@title === 6A: Export Final PLY (~1 min) ===
import os, shutil, glob

OUTPUT_DIR = "/content/gaussian-splatting/output/arena_3dgs"
COLAB_PLY = "/content/arena_3dgs_pointcloud.ply"
DRIVE_PLY = os.path.join(DRIVE_PATH, "arena_3dgs_pointcloud.ply")

# Find the trained point cloud
plys = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.ply")))
if not plys:
    print("No PLY found in output. Train first (Cell 5A or 5B).")
else:
    latest = plys[-1]
    print(f"Latest PLY: {latest} ({os.path.getsize(latest)/1024/1024:.0f} MB)")
    shutil.copy2(latest, COLAB_PLY)
    print(f"Copied to {COLAB_PLY}")
    shutil.copy2(latest, DRIVE_PLY)
    print(f"Copied to {DRIVE_PLY} (Drive backup)")
    from google.colab import files
    files.download(COLAB_PLY)
    print("\nDownload started. Open Cell 6B-6C for validation and compression.")

In [ ]:
#@title === 6B: Validate PLY Format ===
import os
from scripts.train_3dgs_enhanced import validate_pointcloud

COLAB_PLY = "/content/arena_3dgs_pointcloud.ply"
if os.path.exists(COLAB_PLY):
    validate_pointcloud(COLAB_PLY)
else:
    print(f"PLY not found: {COLAB_PLY}. Run Cell 6A first.")

In [ ]:
#@title === 6C: Compress for Local Viewer (~1 min) ===
import os, sys, subprocess, urllib.request

COLAB_PLY = "/content/arena_3dgs_pointcloud.ply"
if not os.path.exists(COLAB_PLY):
    print(f"PLY not found: {COLAB_PLY}. Run Cell 6A first.")
else:
    COMPRESS_PY = "/content/scripts/compress_splat.py"
    if not os.path.exists(COMPRESS_PY):
        print("Downloading compress script...")
        url = ("https://raw.githubusercontent.com/"
               "kaarthik-balakrishnan/arena-3dgs/main/scripts/compress_splat.py")
        urllib.request.urlretrieve(url, COMPRESS_PY)

    for quality in ['medium']:
        out_name = COLAB_PLY.replace('.ply', f'_{quality}.splat')
        !python "{COMPRESS_PY}" "{COLAB_PLY}" --quality {quality} --output "{out_name}"
        if os.path.exists(out_name):
            print(f"Compressed: {out_name} ({os.path.getsize(out_name)/1024/1024:.1f} MB)")
            from google.colab import files
            files.download(out_name)